# PNG padding preview and overwrite

This notebook:
- loads any PNG from a file path
- finds the larger dimension (`width` vs `height`)
- adds **white padding** to make the image square
- lets you preview the result and choose where padding goes
- overwrites the original file when you click **Save / Overwrite original PNG**

**Assumption:** the image is expanded to a square canvas using the image's larger dimension.  
- If the image is **taller than it is wide**, you can place the original on the **left** or **right**.
- If the image is **wider than it is tall**, you can place the original on the **top** or **bottom**.
- If the image is already square, no padding is added.


In [ ]:
from pathlib import Path
from io import BytesIO

from PIL import Image
import ipywidgets as widgets
from IPython.display import display, clear_output


In [ ]:
def make_square_with_white_padding(img: Image.Image, placement: str = "left") -> Image.Image:
    """Return a square image whose size is max(width, height), using white padding.

    placement meanings:
    - when height > width: 'left' puts the image on the left, padding on the right;
                           'right' puts the image on the right, padding on the left.
    - when width > height: 'top' puts the image at the top, padding on the bottom;
                           'bottom' puts the image at the bottom, padding on the top.
    - when already square: returns a copy.
    """
    img = img.convert("RGBA")
    w, h = img.size
    size = max(w, h)

    if w == h:
        return img.copy()

    canvas = Image.new("RGBA", (size, size), (255, 255, 255, 255))

    if h > w:
        # Need horizontal padding.
        if placement not in {"left", "right"}:
            placement = "left"
        x = 0 if placement == "left" else size - w
        y = 0
    else:
        # Need vertical padding.
        if placement not in {"top", "bottom"}:
            placement = "top"
        x = 0
        y = 0 if placement == "top" else size - h

    canvas.paste(img, (x, y), img)
    return canvas


def render_preview(img: Image.Image, max_px: int = 600) -> bytes:
    """Return PNG bytes for notebook display, scaled down for preview if needed."""
    preview = img.copy()
    preview.thumbnail((max_px, max_px))
    bio = BytesIO()
    preview.save(bio, format="PNG")
    return bio.getvalue()


In [ ]:
path_box = widgets.Text(
    value="",
    placeholder="/full/path/to/your/image.png",
    description="PNG path:",
    layout=widgets.Layout(width="700px")
)

load_button = widgets.Button(description="Load PNG", button_style="info")
preview_button = widgets.Button(description="Refresh preview")
save_button = widgets.Button(description="Save / Overwrite original PNG", button_style="warning")

placement_toggle = widgets.ToggleButtons(
    options=[("Left", "left"), ("Right", "right")],
    value="left",
    description="Placement:"
)

info_out = widgets.Output()
image_out = widgets.Output()

state = {
    "path": None,
    "original": None,
    "preview": None,
}


def update_toggle_options(img: Image.Image):
    w, h = img.size
    if h > w:
        placement_toggle.options = [("Left", "left"), ("Right", "right")]
        if placement_toggle.value not in {"left", "right"}:
            placement_toggle.value = "left"
    elif w > h:
        placement_toggle.options = [("Top", "top"), ("Bottom", "bottom")]
        if placement_toggle.value not in {"top", "bottom"}:
            placement_toggle.value = "top"
    else:
        placement_toggle.options = [("Square already", "left")]
        placement_toggle.value = "left"


def show_status(msg: str):
    with info_out:
        clear_output(wait=True)
        print(msg)


def show_preview(*_):
    img = state.get("original")
    if img is None:
        show_status("Load a PNG first.")
        return

    padded = make_square_with_white_padding(img, placement_toggle.value)
    state["preview"] = padded

    with image_out:
        clear_output(wait=True)
        print(f"Original size: {img.size[0]} x {img.size[1]}")
        print(f"Output size:   {padded.size[0]} x {padded.size[1]}")
        display(widgets.HBox([
            widgets.VBox([widgets.HTML("<b>Original</b>"), widgets.Image(value=render_preview(img), format="png")]),
            widgets.VBox([widgets.HTML("<b>Preview</b>"), widgets.Image(value=render_preview(padded), format="png")]),
        ]))


def load_png(_):
    raw = path_box.value.strip().strip('"').strip("'")
    if not raw:
        show_status("Enter a PNG path first.")
        return

    path = Path(raw).expanduser().resolve()
    if not path.exists():
        show_status(f"File not found: {path}")
        return
    if path.suffix.lower() != ".png":
        show_status(f"That file is not a PNG: {path.name}")
        return

    try:
        img = Image.open(path)
        img.load()
    except Exception as e:
        show_status(f"Could not open image: {e}")
        return

    state["path"] = path
    state["original"] = img
    update_toggle_options(img)
    show_status(f"Loaded: {path}")
    show_preview()


def save_overwrite(_):
    path = state.get("path")
    preview = state.get("preview")
    if path is None or preview is None:
        show_status("Load a PNG and generate a preview first.")
        return

    try:
        # Save as PNG back to the exact same path.
        preview.save(path, format="PNG")
        show_status(f"Overwritten successfully: {path}")
    except Exception as e:
        show_status(f"Save failed: {e}")


load_button.on_click(load_png)
preview_button.on_click(show_preview)
save_button.on_click(save_overwrite)
placement_toggle.observe(show_preview, names="value")

ui = widgets.VBox([
    widgets.HBox([path_box, load_button]),
    widgets.HBox([placement_toggle, preview_button, save_button]),
    info_out,
    image_out,
])

display(ui)
show_status("Enter a PNG path, then click 'Load PNG'.")
